# Used Car Price Prediction: Feature Engineering (Brand)

## Notebook Objective

The baseline model in notebook 03 left out the `name` column because it has too many unique values to one-hot encode directly. It scored a validation R² of about 0.49, which means it was **underfitting**.

This notebook tests one change: extracting a **brand** feature from `name` and adding it to the pipeline.

Everything else stays identical to the baseline so the comparison is fair:

- Same dataset, same train-test split (`random_state=42`)
- Same preprocessing approach, same Linear Regression with a log-transformed target
- Same 5-fold cross-validation (`KFold`, `random_state=42`) and the same metrics: MAE, RMSE and R²

**Evaluation protocol:** as before, all comparison is done with 5-fold cross-validation on the training set. The test set is held out and used exactly once, at the end of the project, for the final chosen model.

# 1. Setup

The same libraries and the same cleaned dataset (`car_data_cleaned.csv`) as the baseline notebook are loaded, so any change in performance comes from the new feature and not from different data.

In [25]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

file_path = Path('..') / 'data' / 'car_data_cleaned.csv'
df = pd.read_csv(file_path)
display(df.head())

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,car_age
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner,19
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner,19
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner,14
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner,9
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner,12


# 2. Extracting the Brand Feature

### Creating `brand` from `name`

The `name` column holds the full car name and variant (for example `Hyundai Verna 1.6 SX`), so it has far too many unique values to one-hot encode. The **first word** of each name is the manufacturer, which gives a feature with only a few dozen categories.

```
df["brand"] = df["name"].str.split().str[0]
```

The extraction gives **29 unique brands**, from high-volume ones such as Maruti, Hyundai, Honda, Tata and Mahindra to premium ones such as BMW, Audi, Mercedes-Benz and Jaguar.

**Why brand should help:** brand is a strong price signal. Two cars with identical age, mileage and fuel type can have very different prices depending on whether they are a Maruti or a BMW. The baseline had no way to capture this.

### Limitations of the simple extraction

- **Two-word brands:** `Land Rover` is cut down to `Land`. It is still a consistent label, so it works as a category, but the name is incomplete.
- **Inconsistent names:** `OpelCorsa` appears as one token because the original name is written without a space.
- **Rare brands:** some brands have very few cars (for example Ambassador, Daewoo, Force). Their coefficients will be unreliable, but `handle_unknown='ignore'` ensures a brand that appears only in a validation fold does not cause an error.

These are left as they are for this first version so the effect of the raw brand feature can be measured on its own. Grouping rare brands and fixing two-word names are possible refinements later.

In [26]:
df["brand"] = df["name"].str.split().str[0] 
a = df["brand"].unique()

display(a)

array(['Maruti', 'Hyundai', 'Datsun', 'Honda', 'Tata', 'Chevrolet',
       'Toyota', 'Jaguar', 'Mercedes-Benz', 'Audi', 'Skoda', 'Jeep',
       'BMW', 'Mahindra', 'Ford', 'Nissan', 'Renault', 'Fiat',
       'Volkswagen', 'Volvo', 'Mitsubishi', 'Land', 'Daewoo', 'MG',
       'Force', 'Isuzu', 'OpelCorsa', 'Ambassador', 'Kia'], dtype=object)

# 3. Splitting the Dataset into Training and Test Sets

The split uses the same configuration as the baseline: 80% training, 20% test, `random_state=42`. Because the random state and the data are identical, **the training and test rows are exactly the same as in notebook 03**, which is what makes the comparison valid.

| Dataset | Observations | Columns |
|---|---:|---:|
| `X_train` | 2,861 | 9 |
| `X_test` | 716 | 9 |
| `y_train` | 2,861 | 1 target |
| `y_test` | 716 | 1 target |

`X` now has 9 columns, but the model does not use all of them. `name` and `year` are still present in `X`, and the `ColumnTransformer` below selects only the listed columns and drops the rest by default:

- `name` is replaced by the lower-cardinality `brand`.
- `year` is redundant with `car_age` (`car_age = 2026 - year`), so it is ignored just as in the baseline.

The test set remains untouched.

In [27]:
X = df.drop(columns=['selling_price'])  # Features
y = df['selling_price']  # Target values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_test.shape)
print(X_train.shape)
print(y_test.shape)
print(y_train.shape)

(716, 9)
(2861, 9)
(716,)
(2861,)


# 4. Feature Selection and Preprocessing

The preprocessing is the same as the baseline, with **one change: `brand` is added to the categorical features.**

**1. Categorical features: One-Hot Encoding**

- `fuel`
- `seller_type`
- `transmission`
- `owner`
- `brand` (new)

**2. Numerical features: Standard Scaling**

- `km_driven`
- `car_age`

One-Hot Encoding turns each brand into its own binary column, so the linear model learns a separate price adjustment for each brand. `handle_unknown='ignore'` prevents errors when a validation or test row contains a brand that was not seen during training.

**Data Leakage Prevention:** the preprocessor is part of the pipeline, so the encoder and scaler are fitted only on the training fold during cross-validation. The validation fold never influences preprocessing, and the held-out test set is not touched.

In [28]:
categorical_features = ['fuel', 'seller_type', 'transmission', 'owner', 'brand']
numerical_features = ['km_driven', 'car_age']

preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('numerical', StandardScaler(), numerical_features)
    ]
)

# 5. Pipeline and Log-Transformed Target

The pipeline is unchanged from the baseline: the preprocessor followed by `LinearRegression`, wrapped in `TransformedTargetRegressor` with `log1p` / `expm1`.

`selling_price` is right-skewed, so the model is trained on `log1p(selling_price)` and its predictions are converted back with `expm1`. All metrics are therefore still reported in the original price scale (₹), which keeps them directly comparable with notebook 03.

Keeping the model, target transformation and metrics identical means that **brand is the only thing that changed**, so any improvement can be credited to it.

In [29]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

cv_pipeline = TransformedTargetRegressor(
    regressor=model_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

# 6. Cross-Validation Setup

`KFold(n_splits=5, shuffle=True, random_state=42)` is used again, so the folds are identical to the baseline notebook. The metrics are MAE, RMSE and R² (in ₹).

As before, the `test_*` keys returned by `cross_validate` are **validation-fold** scores, not scores on the held-out test set.

In [30]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'MAE': 'neg_mean_absolute_error',
    'RMSE': 'neg_root_mean_squared_error',
    'R2': 'r2'
}

# 7. Dummy Baseline (5-Fold CV)

The dummy model again predicts the training-fold median for every car. It is re-run here so that the dummy and the new model are compared in the same notebook, on the same folds.

Its validation R² is slightly negative (about -0.06), meaning it does slightly worse than predicting the mean of each validation fold, which is expected for a model that ignores every feature.

In [31]:
dummy_results = cross_validate(
    estimator=DummyRegressor(strategy='median'),
    X=X_train,
    y=y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

# 8. Linear Regression with Brand: 5-Fold Cross-Validation

### 5-Fold CV Results (Linear Regression + brand, log-target)

| Metric | Train (mean) | Validation (mean ± std) |
|---|---:|---:|
| MAE | ₹1.33L | ₹1.37L ± 0.14L |
| RMSE | ₹2.60L | ₹2.69L ± 0.58L |
| R² | 0.722 | 0.698 ± 0.081 |

**Gaps:** MAE gap ≈ +₹3,503, RMSE gap ≈ +₹9,265, R² gap ≈ +0.024.

**Interpretation:**

- **Underfitting is much reduced:** validation R² rose from about 0.49 to about 0.70, so the model now explains roughly 70% of the price variation instead of about half. Brand was a major missing signal.
- **A small train-validation gap has appeared:** in the baseline the gaps were almost zero, and now all three are slightly positive. The model has about 29 extra columns, so it can fit the training folds a little more closely. The gap is still small, so this is mild at most and not a real overfitting problem.
- **Remaining error:** an R² of about 0.70 means there is still a lot of unexplained variation, so other features or a more flexible model could still help.

In [32]:
cv_results = cross_validate(
    estimator=cv_pipeline,
    X=X_train,
    y=y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

cv_summary = pd.DataFrame({
    'Fold': range(1, kf.get_n_splits() + 1),

    'Train_MAE': -cv_results['train_MAE'],
    'Val_MAE': -cv_results['test_MAE'],

    'Train_RMSE': -cv_results['train_RMSE'],
    'Val_RMSE': -cv_results['test_RMSE'],

    'Train_R2': cv_results['train_R2'],
    'Val_R2': cv_results['test_R2']
})

display(cv_summary)

summary = cv_summary.drop(columns='Fold').agg(['mean', 'std']).T
display(summary)

# Gaps: for all three, a large positive value suggests overfitting
mae_gap = cv_summary['Val_MAE'].mean() - cv_summary['Train_MAE'].mean()
rmse_gap = cv_summary['Val_RMSE'].mean() - cv_summary['Train_RMSE'].mean()
r2_gap = cv_summary['Train_R2'].mean() - cv_summary['Val_R2'].mean()

print(f'MAE gap  (Val - Train): {mae_gap:,.0f}')
print(f'RMSE gap (Val - Train): {rmse_gap:,.0f}')
print(f'R2 gap   (Train - Val): {r2_gap:.4f}')

,Fold,Train_MAE,Val_MAE,Train_RMSE,Val_RMSE,Train_R2,Val_R2
0,1,131000.752176,142419.857272,230585.128177,365318.197055,0.766512,0.565029
1,2,135807.670602,127990.198388,273209.039427,222509.413361,0.704538,0.761117
2,3,136055.396029,128212.247106,268010.155134,242436.222588,0.703884,0.762207
3,4,136096.562416,127531.956353,272066.792182,233301.860573,0.715227,0.692787
4,5,128164.068095,158485.973506,256639.336334,283270.131123,0.719489,0.709379


,mean,std
Train_MAE,133424.889864,3649.922265
Val_MAE,136928.046525,13592.654270
Train_RMSE,260102.090251,17753.347670
Val_RMSE,269367.164940,58355.641867
Train_R2,0.721930,0.025819
Val_R2,0.698104,0.080536


MAE gap  (Val - Train): 3,503
RMSE gap (Val - Train): 9,265
R2 gap   (Train - Val): 0.0238


In [33]:
def run_cv(cat_cols, num_cols):
    pre = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ])
    model = TransformedTargetRegressor(
        regressor=Pipeline([('pre', pre), ('lr', LinearRegression())]),
        func=np.log1p, inverse_func=np.expm1
    )
    return cross_validate(model, X_train, y_train, cv=kf, scoring=scoring)

base_cat = ['fuel', 'seller_type', 'transmission', 'owner']
no_brand = run_cv(base_cat, numerical_features)
with_brand = run_cv(base_cat + ['brand'], numerical_features)

mae_gain = with_brand['test_MAE'] - no_brand['test_MAE'] 
print(f"MAE improvement per fold: {mae_gain.round(0)}")
print(f"Mean ± std: {mae_gain.mean():,.0f} ± {mae_gain.std(ddof=1):,.0f}")
print(f"Folds improved: {(mae_gain > 0).sum()}/5")

MAE improvement per fold: [32909. 36647. 39743. 30968. 32549.]
Mean ± std: 34,563 ± 3,567
Folds improved: 5/5


## Ablation Study: Evaluating the Impact of Brand

To determine whether the car brand improves prediction accuracy, I performed an **ablation study** by comparing two model configurations using 5-fold cross-validation.

### Results

The difference in MAE between the two models was calculated for each of the five validation folds.

| Fold | MAE Improvement |
|---|---:|
| 1 | 32,909 |
| 2 | 36,647 |
| 3 | 39,743 |
| 4 | 30,968 |
| 5 | 32,549 |

**Summary:**

- **Mean MAE Improvement:** 34,563
- **Standard Deviation:** 3,567
- **Folds Improved:** 5/5

### 3. Key Observations

- Adding the `brand` feature improved prediction performance across all five validation folds.
- All five folds showed improvement, suggesting that the benefit was consistent across different validation splits.

### 4. Conclusion

The ablation study demonstrates that including `brand` improves the model's predictive performance.

Since the model improved in all five cross-validation folds, I decided to retain `brand` as one of the features for subsequent model development.

# 9. Dummy vs Linear Regression with Brand vs Baseline

### Comparison with the baseline (validation, 5-fold CV)

| Model | Val MAE | Val RMSE | Val R² |
|---|---:|---:|---:|
| Dummy (median) | ₹2.73L | ₹5.06L | -0.063 |
| Linear Regression, baseline (no brand) | ₹1.71L | ₹3.51L | 0.491 |
| Linear Regression + brand | ₹1.37L | ₹2.69L | 0.698 |

### Observations

- Adding brand cuts the validation MAE by about **20%** compared with the baseline model (₹1.71L to ₹1.37L), and by about **50%** compared with the dummy.
- RMSE falls from about ₹3.51L to ₹2.69L, so large errors on expensive cars also shrink.
- R² improves by about **+0.21** (0.49 to 0.70), which is a large gain from a single feature.
- The fold-to-fold spread of RMSE is still large (±₹0.58L), so the improvement should be read as an average across folds rather than a guarantee for every car.
- All numbers come from the same 5 folds on the training set, so the comparison is fair and the test set remains untouched.

In [34]:
def summarize(name, res):
    return {
        'Model': name,
        'Val_MAE': -res['test_MAE'].mean(),
        'Val_MAE_std': res['test_MAE'].std(ddof=1),
        'Val_RMSE': -res['test_RMSE'].mean(),
        'Val_RMSE_std': res['test_RMSE'].std(ddof=1),
        'Val_R2': res['test_R2'].mean(),
        'Val_R2_std': res['test_R2'].std(ddof=1),
    }

comparison = pd.DataFrame([
    summarize('Dummy (median)', dummy_results),
    summarize('Linear Regression (Brand)', cv_results),
])
display(comparison.round(3))

mae_reduction = (1 - comparison.loc[1, 'Val_MAE'] / comparison.loc[0, 'Val_MAE']) * 100
print(f'Validation MAE reduction vs dummy: {mae_reduction:.2f}%')

,Model,Val_MAE,Val_MAE_std,Val_RMSE,Val_RMSE_std,Val_R2,Val_R2_std
0,Dummy (median),273095.826,21351.390,505761.755,56746.683,-0.063,0.023
1,Linear Regression (Brand),136928.047,13592.654,269367.165,58355.642,0.698,0.081


Validation MAE reduction vs dummy: 49.86%


In [35]:
from sklearn.model_selection import cross_val_predict

pred = cross_val_predict(cv_pipeline, X_train, y_train, cv=kf)
err = pd.DataFrame({
    'brand': X_train['brand'], 'actual': y_train, 'pred': pred
})
err['abs_err'] = (err.actual - err.pred).abs()
err['ape'] = err.abs_err / err.actual

display(err.groupby('brand').agg(n=('ape', 'size'),
                               MAE=('abs_err', 'mean'),
                               MdAPE=('ape', 'median'))
         .sort_values('MAE', ascending=False).head(10))
print("Overall MdAPE:", err.ape.median())

,n,MAE,MdAPE
brand,,,
Volvo,4,1.219178e+06,0.317570
Land,5,1.168282e+06,0.261520
Jaguar,5,1.071000e+06,0.475459
Audi,26,7.360078e+05,0.235656
BMW,22,5.838117e+05,0.240316
Mercedes-Benz,13,5.082795e+05,0.190773
Force,1,3.281204e+05,0.948325
Toyota,132,3.011480e+05,0.320436
Ford,171,1.788428e+05,0.290455


Overall MdAPE: 0.24139641967396927


## Brand-wise Error Analysis

To understand how well the model predicts car prices across different brands, I performed a brand-wise error analysis using **cross-validation predictions**.

###  Key Observations

- **Luxury brands:** Volvo, Jaguar, Audi, and BMW have relatively high MAE values. Higher-priced vehicles can produce larger absolute prediction errors.
- **Jaguar:** Has a MdAPE of approximately 47.55%, indicating substantial relative prediction errors.
- **Force:** Has the highest MdAPE at 94.83%, but this result is based on only one observation and should not be treated as reliable evidence of poor model performance.
- **Toyota:** Despite having 132 observations, its MdAPE is approximately 32.04%, suggesting room for improvement in relative prediction accuracy.
- **Mercedes-Benz:** Has a relatively lower MdAPE of 19.08% among the listed brands.

###  Conclusion

The model's prediction errors vary across car brands. Luxury brands tend to have higher absolute errors, while some brands also show considerable percentage errors.

However, results for brands with very few observations should be interpreted cautiously. Both MAE and MdAPE are useful here: MAE measures errors in actual price units, while MdAPE helps compare relative prediction performance across brands.2

# 10. Testing on Test set

In [ ]:
cv_pipeline.fit(X_train, y_train)
pred_test = cv_pipeline.predict(X_test)

print(f"Test MAE : {mean_absolute_error(y_test, pred_test):,.0f}")
print(f"Test RMSE: {root_mean_squared_error(y_test, pred_test):,.0f}")
print(f"Test R2  : {r2_score(y_test, pred_test):.3f}")
print(f"Test MdAPE: {(np.abs(y_test - pred_test) / y_test).median():.3f}")

Test MAE : 144,221
Test RMSE: 304,582
Test R2  : 0.712
Test MdAPE: 0.249


# 11. Conclusion

## Objective and approach

This notebook tested one change to the baseline from notebook 03: extracting `brand` from `name` and adding it to a Linear Regression with a log-transformed target. Everything else (data, split, folds, metrics) was kept identical, so any difference can be credited to `brand`.

## Results

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Dummy (median), CV | ₹2.73L | ₹5.06L | -0.063 |
| Linear Regression, no brand, CV | ₹1.71L | ₹3.51L | 0.491 |
| Linear Regression + brand, CV | ₹1.37L | ₹2.69L | 0.698 |
| **Linear Regression + brand, held-out test** | **₹1.44L** | **₹3.05L** | **0.712** |

## Key findings

- **Brand clearly helps.** The ablation showed a mean MAE improvement of ₹34,563 ± ₹3,567, with brand better in 5 of 5 folds. The gain is much larger than the fold-to-fold variation, so it is not noise.
- **The model is not overfitting.** Train and validation scores are close (R² gap ≈ 0.024), so the remaining error comes from missing signal, not from excess model complexity.
- **Test performance matches cross-validation.** Test R² is 0.712 against a CV R² of 0.698. Test MAE (₹1.44L) is about 5% above the CV MAE (₹1.37L), and test RMSE (₹3.05L) is within the fold-to-fold spread of the CV RMSE (₹2.69L ± ₹0.58L). The test set was used once, for the final model only, with no tuning afterwards.
- **Typical error is about 25% of the price.** Test MdAPE is 0.249, close to the CV value of 0.241. The percentage error is similar across most brands, so the large absolute errors on Audi, BMW, Volvo and Jaguar mostly reflect their higher prices.
- **Toyota is the clearest weak spot among high-volume brands** (132 cars, MdAPE ≈ 32%). Toyota sells both cheap hatchbacks and expensive SUVs, so brand alone is too coarse for it.

## Limitations

- Only linear models were tested, so non-linear effects (for example, age and mileage interacting with brand) are not captured.
- `brand` is a coarse feature, and the first-word extraction splits or mislabels some brands (`Land Rover` becomes `Land`, `OpelCorsa` stays as one token).
- Several brands have very few cars, so their error estimates are unreliable.
- The dataset has about 3.5k rows, and the test set (716 cars) gives only a rough estimate of real-world accuracy.